# Development of Arctic boundary current model

See `gridap_development/ArcticBoundaryCurrentModel12.ipynb` for development of NS solver using `gridap`. This version switches to `nuPGCM`.

13.  `ArcticBoundaryCurrentModel13.ipynb`: Migrate to [`nuPGCM`](https://github.com/hgpeterson/nuPGCM) by merging `ArcticBoundaryCurrentModel12.ipynb` with `nuPGCM/examples.run.jl`.
14.  `ArcticBoundaryCurrentModel14.ipynb`: Switch to `ArcticBasin.msh` geometry.
15.  `ArcticBoundaryCurrentModel15.ipynb`: Make consistent with `examples/run.jl`.
16.  `ArcticBoundaryCurrentModel16.ipynb`: Adjust plotting. Experiment with different parameter settings.
17.  `ArcticBoundaryCurrentModel17.ipynb`: Add forcing from v0.4.0 and update from my customizations (0.3.0_custom).

Goals: Add wind forcing, buoyancy forcing, and T/S equation of state

Run this notebook with `nuPGCM` branch `0.4.0_custom`

twnh August '25

## Problem statement

Following Peterson & Callies (2025, in review at JPO, their (A1)–(A4), (13)), the (non-dimensional) equations we solve are:
$$
\text{Inversion: } 
\left\lbrace
\begin{aligned}
-v &= - \frac{\partial p}{\partial x} + \epsilon^2 \left( \alpha^2 \frac{\partial ^2 u}{\partial x^2} + \alpha^2 \frac{\partial ^2 u}{\partial y^2} + \frac{\partial ^2 u}{\partial z^2}\right), \\
u &= - \frac{\partial p}{\partial y} + \epsilon^2 \left( \alpha^2 \frac{\partial ^2 v}{\partial x^2} + \alpha^2 \frac{\partial ^2 v}{\partial y^2} + \frac{\partial ^2 v}{\partial z^2}\right), \\
\frac{\partial p}{\partial z} & = b + \epsilon^2 \alpha^2 \left( \alpha^2 \frac{\partial ^2 w}{\partial x^2} + \alpha^2 \frac{\partial ^2 w}{\partial y^2} + \frac{\partial ^2 w}{\partial z^2} \right) ,\\
\nabla\cdot \boldsymbol{u} &= 0,
\end{aligned}
\right.
$$
$$
\text{Buoyancy: } 
\mu \varrho \left( \frac{\partial b}{\partial t} + \boldsymbol{u} \cdot \nabla b \right) = \epsilon^2 \alpha^2 \left[ \frac{\partial}{\partial x} \left( \kappa(\boldsymbol{x}) \frac{\partial b}{\partial x}\right) + \frac{\partial}{\partial y} \left( \kappa (\boldsymbol{x}) \frac{\partial b}{\partial y}\right) + \frac{\partial}{\partial z} \left( \kappa(\boldsymbol{x}) \frac{\partial b}{\partial z}\right) \right]
$$
(see also Peterson & Callies, 2025, in prep. for JAMES, their (12)–(14)).
Forcing is given by:
$$
\text{Boundary conditions: }\left\lbrace
\begin{aligned}
\boldsymbol{t} \cdot \boldsymbol{\sigma} \cdot \mathbf{n} = \tau_{\text{imposed}} (\boldsymbol{x}) &\text{ on } \Gamma_s, \\
\boldsymbol{u} = \boldsymbol{0} &\text{ on } \Gamma_w . \\
\end{aligned}
\right.
$$
However, their solutions are unforced, and the `examples/run.jl` code has no forcing.
The domain $\Omega$ rotates with constant Coriolis parameter and has boundary $\partial \Omega$, ${\mathbf n}$ is the unit outward normal, $\boldsymbol{t}$ is the tangential direction at the surface, $\Gamma_s$, and $\boldsymbol{\sigma} = \nabla u$ is the stress vector. The driving force is the tangential stress $\tau_{\text{imposed}}$ (units of $\text{m}^{2} \text{s}^{-2}$ ). The mean value of the pressure anomaly is constrained to equal zero,
$$
\int_\Omega p \ {\rm d}\Omega = 0 .
$$
The initial condition is no flow and $b_{tot} = b + N^2 z$, so that $b$ measures the anomaly from uniform stratification (see: `nuPGCM/src/model.jl` and this [issue](https://github.com/hgpeterson/nuPGCM/issues/5).)

We're also interested in:
1. Two-component equation of state for buoyancy $b$.
2. Buoyancy forcing on $\Gamma_s$ and $\Gamma_w$.
3. Free-slip (no stress) boundary conditions on $\Gamma_w$ (but not for the stress-driven case, unless the integrated applied stress vanishes).

### Non-dimensional parameters

The scaling is as follows (Peterson & Callies, 2025, in review):
$$
\begin{align}
x, y & \sim L, \nonumber \\
z & \sim H_0, \nonumber \\
u, v & \sim U_0, \nonumber \\
w & \sim \frac{U_0 H_0}{L}, \nonumber \\
f & \sim f_0, \nonumber \\
\nu & \sim \nu_0, \nonumber \\
\kappa & \sim \kappa_0 ,\nonumber \\
p & \sim f_0 U_0 L, \text{ with units of Pa/(kgm$^{-3}$)}\nonumber \\
b & \sim \frac{f_0 U_0 L}{H_0} = N^2 H_0  \implies U_0 = \frac{N^2 H_0^2}{f_0 L} , \nonumber \\
t & \sim \frac{L}{U_0} , \nonumber \\
% \tau \text{ (units m}^2\text{s}^2\text{)} & \sim \frac{\nu_0 U_0}{H_0} , \nonumber 
\end{align}
$$
with non-dimensional numbers
$$
\begin{align}
\text{Ekman number: } \epsilon & = \frac{\nu_0} {f_0 H_0^2} , \nonumber \\
\text{Burger number: } \varrho & = \frac{N^2 H_0^2} {f_0^2 L^2} ,  \nonumber \\
\text{Turbulent Prandtl number: } \mu & = \frac{\nu_0} {\kappa_0} ,  \nonumber \\
\text{Aspect ratio: } \alpha & = \frac{H_0} {L}
\end{align}
$$

The dimensional viscosity is a constant $\nu = \nu_0$, but the diffusivity varies in space, usually decaying away from the bottom $\kappa(z)$.

We specify:
\begin{align}
\text{Characteristic domain depth:  } & H_0, \nonumber\\
\text{Characteristic domain width:  } & L, \nonumber\\
\text{Coriolis parameter (uniform): } & f_0,\nonumber\\
\text{Characteristic diffusivity:   } & \kappa_0, \nonumber\\
\text{Characteristic viscosity (uniform): } & \nu_0, \nonumber\\
\text{Characteristic buoyancy frequency: } & N,  \nonumber\\
%\text{Characteristic surface stress:   } & \tau_0, \nonumber\\
\text{Non-dimensional diffusivity function: } & \kappa (\boldsymbol{x}), \nonumber\\
\text{Non-dimensional domain depth function: } & H (\boldsymbol{x}), \nonumber\\
%\text{Non-dimensional surface stress function: } & \tau (\boldsymbol{x}), \nonumber\\
\text{Initial non-dimensional buoyancy field: } & b_0 (\boldsymbol{x}), \nonumber\\
\end{align}

#### Import packages

In [1]:
using nuPGCM
using JLD2
using LinearAlgebra
using Printf
# using PyPlot
using Revise

#### Configure nuPGCM

In [2]:
# pygui(false)
# plt.style.use(joinpath(@__DIR__, "plots.mplstyle"))
# plt.close("all")

set_out_dir!(joinpath(@__DIR__, ""))

# architecture and dimension
arch = CPU() ;

┌ Info: Output directory set to '/Users/twnh/Library/CloudStorage/OneDrive-JohnsHopkins/Documents/Meetings/ArcticOceanDynamics_Nov25/ArcticBoundaryCurrentModel/nuPGCM/'
└ @ nuPGCM /Users/twnh/Library/CloudStorage/OneDrive-JohnsHopkins/Documents/Meetings/ArcticOceanDynamics_Nov25/ArcticBoundaryCurrentModel/nuPGCM/src/nuPGCM.jl:31


#### Define physical parameters

In [3]:
# Realistic parameters
# L  = 1000.0e3   # (m) domain size
# H₀ = 1000.0     # (m) domain height
# f₀ = 2.0e-4     # (s⁻¹) Coriolis parameter
# ν₀ = 8.0e-2     # (m²/s) kinematic viscosity
# κ₀ = 8.0e-2     # (m²/s) kinematic diffusivity
# N² = (10*f₀)^2  # (s⁻²) buoyancy frequency squared

# Values consistent with examples/run.jl
L  = 1000.0e3               # (m) domain size
H₀ = L/2                    # (m) domain height
f₀ = 2.0e-4                 # (s⁻¹) Coriolis parameter
ν₀ = 0.1^2 * f₀ * H₀^2      # (m²/s) kinematic viscosity
κ₀ = ν₀                     # (m²/s) kinematic diffusivity
N² = (2*f₀)^2               # (s⁻²) buoyancy frequency squared

# Experiment with different parameters
H₀ = L/4                    # (m) domain height
N² = (10*f₀)^2              # (s⁻²) buoyancy frequency squared

# Beta-effect
β = 0.0 # Set to zero for an f-plane

# Wind forcing parameters
ρₒ = 1000.0  # (kg/m³) reference density
# cᴰ = 2.5e-3  # dimensionless drag coefficient
# ρₐ = 1.225   # (kg/m³) average density of air
# function u₁₀(x_in)
#     x,y = x_in[1], x_in[2]
#     r = sqrt(x^2 + y^2)
#     θ = atan(y, x)
#     radial_wind = 1.0e-1 * r        # Set magnitude of wind velocity here
#     u₁₀ = radial_wind * [ - sin(θ),  cos(θ), 0.0 ] # m s⁻¹
#     return u₁₀
# end

# τ(x) = VectorValue((ρₐ / ρₒ) * cᴰ * u₁₀(x) * norm(u₁₀(x))) # m² s⁻²
# uₛ = sqrt((ρₐ / ρₒ) * cᴰ) # m/s, friction velocity  
# println("Friction velocity: ", uₛ, " m/s")
# U₀ = uₛ^2 * H₀ / ν₀   # (m/s) speed scale
U₀ = N² * H₀^2 / (f₀ * L)   # (m/s) speed scale
println("Speed scale                       : ", U₀, " m/s")
println("Time scale                        : ", L/U₀, " s")
println("Pressure scale                    : ", f₀*U₀*L*ρₒ, " Pa")

Ekman_layer_depth = sqrt(2*ν₀ / f₀) # (m), Ekman layer depth
println("Ekman layer depth                 : ", Ekman_layer_depth, " m")
println("Non-dimensional Ekman layer depth : ", Ekman_layer_depth/H₀)

ε = sqrt(ν₀ / (f₀ * H₀^2))      # Ekman number
println("\nEkman number ε             : ", ε)
ϱ = (N² * H₀^2) / (f₀^2 * L^2)  # Burger number
println("Burger number ϱ            : ", ϱ)
μ = ν₀ / κ₀                     # Turbulent Prandtl number
println("Turbulent Prandtl number μ : ", μ)
α = H₀ / L                     # Aspect ratio
println("Domain aspect ratio α      : ", α)

Speed scale                       : 1250.0 m/s
Time scale                        : 800.0 s
Pressure scale                    : 2.5e8 Pa
Ekman layer depth                 : 70710.67811865476 m
Non-dimensional Ekman layer depth : 0.28284271247461906

Ekman number ε             : 0.2
Burger number ϱ            : 6.25
Turbulent Prandtl number μ : 1.0
Domain aspect ratio α      : 0.25


#### Compute numerical parameters

In [4]:
μϱ = μ * ϱ  
Δt = 1e-5*μϱ/ε^2/α^2            # Time step size
println("Non-dimensional time step size            : ", Δt)

# Non-dimensional Coriolis parameter
f(x) = 1 + β*x[2]

# Non-dimensional viscosity
ν(x) = 1  # viscosity

# Final time of integration
# Tf = 1e-2*μϱ/ε^2/α^2  # As in examples/run.jl
Tf = 0.1*μϱ/ε^2  # simulation time
# Tf = Δt           # Just one step.
println("Non-dimensional final time of integration : ", Tf)
println("Number of steps                           : ", Tf/Δt)

Non-dimensional time step size            : 0.024999999999999994
Non-dimensional final time of integration : 15.624999999999996
Number of steps                           : 625.0


### Build and load mesh

In [5]:
if(false)       # Use Tom's 2D mesh generation function.  DOESN'T WORK YET!
    mesh_name = "meshes/ArcticBasin2D.msh"
    dim = 2
    hh = 2e-2   # Sets mesh spacing
    H, dH, R, r = generate_arctic_basin2D( hh, α; show_gui=false, savefile=mesh_name, DEBUG=false)
elseif(false)   # Use Tom's 3D mesh generation function.    DOESN'T WORK YET!
                # This option takes a while to run (like an hour on my MacBook), and it gives very similar results to the 2D case.
    include("meshes/mesh_ArcticBasin3D.jl")
    mesh_name = "meshes/ArcticBasin3D.msh"
    dim = 3
    hh = 4e-2   # Sets mesh spacing
    H, dH, R, r = generate_arctic_basin3D( hh, α; show_gui=false, savefile=mesh_name, DEBUG=false)
elseif(false)    # Use Henry's original 2D parabolic domain and mesh, modified with Tom's tags in the new spaces.jl file in nuPGCM/src.    DOESN'T WORK YET!
    dim = 2
    hh = 1e-2   # Sets mesh spacing
    include("meshes/mesh_bowl2D.jl")
    mesh_name = joinpath(@__DIR__, @sprintf("meshes/bowl2D_%e_%e.msh", hh, α))
elseif(true)   # Use Henry's original 3D parabolic domain and mesh, modified with Tom's tags in the new spaces.jl file in nuPGCM/src.
                # This option takes a while to run, and it gives very similar results to the 2D case.
    dim = 3
    hh = 8e-2   # Sets mesh spacing
    include("meshes/mesh_bowl3D.jl")
    mesh_name = @sprintf("bowl%dD_%e_%e", dim, h, α)
    
end # if
@info "Loading mesh from file: " mesh_name
mesh = Mesh(joinpath(@__DIR__, "meshes/$mesh_name.msh")) ;

┌ Info: 2εₘᵢₙ = 2h/(α√2) = 2.3e-01
└ @ Main /Users/twnh/Library/CloudStorage/OneDrive-JohnsHopkins/Documents/Meetings/ArcticOceanDynamics_Nov25/ArcticBoundaryCurrentModel/nuPGCM/meshes/mesh_bowl3D.jl:51
┌ Info: Loading mesh from file: 
│   mesh_name = bowl3D_8.000000e-02_5.000000e-01
└ @ Main /Users/twnh/Library/CloudStorage/OneDrive-JohnsHopkins/Documents/Meetings/ArcticOceanDynamics_Nov25/ArcticBoundaryCurrentModel/nuPGCM/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X16sZmlsZQ==.jl:26


Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Bezier)
Info    : [ 70%] Meshing curve 3 (Circle)
Info    : Done meshing 1D (Wall 0.00134325s, CPU 0.001333s)
Info    : Meshing 2D...
Info    : [  0%] Meshing surface 1 (Surface of Revolution, Frontal-Delaunay)
Info    : [ 40%] Meshing surface 2 (BSpline surface, Frontal-Delaunay)
Info    : [ 70%] Meshing surface 3 (Surface of Revolution, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.0748415s, CPU 0.074522s)
Info    : Meshing 3D...
Info    : 3D Meshing 1 volume with 1 connected component
Info    : Tetrahedrizing 1354 nodes...
Info    : Done tetrahedrizing 1362 nodes (Wall 0.00622646s, CPU 0.006166s)
Info    : Reconstructing mesh...
Info    :  - Creating surface mesh
Info    :  - Identifying boundary edges
Info    :  - Recovering boundary
Info    : Done reconstructing mesh (Wall 0.0142582s, CPU 0.014063s)
Info    : Found volume 1
Info    : It. 0 - 0 nodes created - worst tet radius 3.19258 (nodes removed 0 0)
Info    : It. 

┌ Info: Boundary labels: 
│   label_names = ["Bottom", "coastline", "Surface", "interior"]
└ @ nuPGCM /Users/twnh/Library/CloudStorage/OneDrive-JohnsHopkins/Documents/Meetings/ArcticOceanDynamics_Nov25/ArcticBoundaryCurrentModel/nuPGCM/src/meshes.jl:19


In [6]:
@warn "Make sure the mesh geometry is consistent with the depth(x) definition (see domain_depth() function in meshes/mesh_ArcticBasin3D.jl)."
# Non-dimensional diffusivity (enhanced at the bottom). See Peterson & Callies (2025) eq. (17)
mixing_depth = 0.1*α  # Depth over which diffusivity increases near the bottom
@info "Mixing depth (non-dimensional): " mixing_depth
κ(x) = 1e-2 + exp(-(x[3] + depth(x))/mixing_depth) ;

τˣ(x) = 0  # zonal wind stress
τʸ(x) = 0  # meridional wind stress
b₀(x) = 0  # surface buoyancy boundary condition
force_build_inversion = true
force_build_evolution = true

params = Parameters(ε, α, μϱ, N²/(N² * α), Δt) ;       # Non-dimensional parameters, including N²/N² for consistency with nuPGCM, which is entirely non-dimensional.

┌ Warning: Make sure the mesh geometry is consistent with the depth(x) definition (see domain_depth() function in meshes/mesh_ArcticBasin3D.jl).
└ @ Main /Users/twnh/Library/CloudStorage/OneDrive-JohnsHopkins/Documents/Meetings/ArcticOceanDynamics_Nov25/ArcticBoundaryCurrentModel/nuPGCM/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X20sZmlsZQ==.jl:1
┌ Info: Mixing depth (non-dimensional): 
│   mixing_depth = 0.05
└ @ Main /Users/twnh/Library/CloudStorage/OneDrive-JohnsHopkins/Documents/Meetings/ArcticOceanDynamics_Nov25/ArcticBoundaryCurrentModel/nuPGCM/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X20sZmlsZQ==.jl:4


### Build system

In [7]:
# FE data
spaces = Spaces(mesh, b₀)
fe_data = FEData(mesh, spaces)
@info "DOFs: $(fe_data.dofs.nu + fe_data.dofs.nv + fe_data.dofs.nw + fe_data.dofs.np)"

┌ Info: DOFs: 31447
└ @ Main /Users/twnh/Library/CloudStorage/OneDrive-JohnsHopkins/Documents/Meetings/ArcticOceanDynamics_Nov25/ArcticBoundaryCurrentModel/nuPGCM/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X22sZmlsZQ==.jl:4


#### Build inversion matrices if necessary: `A_inversion`, `B_inversion`

To be sure, delete the saved `.jld2` files. Sometimes the file exists, but for a different parameter set, which leads to inconsistencies and errors.

In [8]:
if !isdir(joinpath(@__DIR__, "matrices"))
    @info "Creating matrices directory"
    mkdir(joinpath(@__DIR__, "matrices"))
end
A_inversion_fname = joinpath(@__DIR__, @sprintf("matrices/A_inversion_%sD_%e_%e_%e_%e_%e.jld2", dim, hh, ε, α, f₀, β))

if force_build_inversion
    @warn "You set `force_build_inversion` to `true`, building matrices..."
    A_inversion, B_inversion, b_inversion = build_inversion_matrices(fe_data, params, f, ν, τˣ, τʸ; A_inversion_ofile=A_inversion_fname)
elseif !isfile(A_inversion_fname)
    @warn "A_inversion file not found, generating..."
    A_inversion, B_inversion, b_inversion = build_inversion_matrices(fe_data, params, f, ν, τˣ, τʸ; A_inversion_ofile=A_inversion_fname)
else
    file = jldopen(A_inversion_fname, "r")
    A_inversion = file["A_inversion"]
    close(file)
    B_inversion = nuPGCM.build_B_inversion(fe_data, params)
    b_inversion = nuPGCM.build_b_inversion(fe_data, params, τˣ, τʸ)
end
nothing

┌ Warning: You set `force_build_inversion` to `true`, building matrices...
└ @ Main /Users/twnh/Library/CloudStorage/OneDrive-JohnsHopkins/Documents/Meetings/ArcticOceanDynamics_Nov25/ArcticBoundaryCurrentModel/nuPGCM/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X24sZmlsZQ==.jl:8


build A_inversion: 10.320876 seconds (20.10 M allocations: 1.143 GiB, 0.87% gc time, 93.84% compilation time)


┌ Info: A_inversion saved to '/Users/twnh/Library/CloudStorage/OneDrive-JohnsHopkins/Documents/Meetings/ArcticOceanDynamics_Nov25/ArcticBoundaryCurrentModel/nuPGCM/matrices/A_inversion_3D_8.000000e-02_2.000000e-01_5.000000e-01_2.000000e-04_0.000000e+00.jld2' (0.026 GB)
└ @ nuPGCM /Users/twnh/Library/CloudStorage/OneDrive-JohnsHopkins/Documents/Meetings/ArcticOceanDynamics_Nov25/ArcticBoundaryCurrentModel/nuPGCM/src/matrices.jl:83


#### Re-order dofs

In [9]:
A_inversion = A_inversion[fe_data.dofs.p_inversion, fe_data.dofs.p_inversion]
B_inversion = B_inversion[fe_data.dofs.p_inversion, :]
b_inversion = b_inversion[fe_data.dofs.p_inversion] ;

#### Preconditioner `P_inversion`

In [10]:
if typeof(arch) == CPU
    @time "lu(A_inversion)" P_inversion = lu(A_inversion)
else
    P_inversion = Diagonal(on_architecture(arch, 1/hh^dim*ones(size(A_inversion, 1))))
end
nothing

lu(A_inversion): 1.039623 seconds (526.36 k allocations: 814.771 MiB, 1.43% gc time, 8.22% compilation time)


#### Move matrices to architecture

In [11]:
A_inversion = on_architecture(arch, A_inversion)
B_inversion = on_architecture(arch, B_inversion)
b_inversion = on_architecture(arch, b_inversion) ;

#### Setup inversion toolkit `inversion_toolkilt`

In [12]:
inversion_toolkit = InversionToolkit(A_inversion, P_inversion, B_inversion, b_inversion; atol=1e-6, rtol=1e-6) ;

#### Build evolution matrices `A_adv`, `A_diff`, `B_diff`, `b_diff` and test against saved matrices

In [13]:
A_adv, A_diff, B_diff, b_diff = build_evolution_system(fe_data, params, κ;
                                    force_build=force_build_evolution,
                                    filename=joinpath(@__DIR__, "matrices/evolution_$mesh_name.jld2")) ;

┌ Warning: `force_build` set to `true`, building evolution system...
│   filename = /Users/twnh/Library/CloudStorage/OneDrive-JohnsHopkins/Documents/Meetings/ArcticOceanDynamics_Nov25/ArcticBoundaryCurrentModel/nuPGCM/matrices/evolution_bowl3D_8.000000e-02_5.000000e-01.jld2
└ @ nuPGCM /Users/twnh/Library/CloudStorage/OneDrive-JohnsHopkins/Documents/Meetings/ArcticOceanDynamics_Nov25/ArcticBoundaryCurrentModel/nuPGCM/src/matrices.jl:160
┌ Info: Evolution system saved to '/Users/twnh/Library/CloudStorage/OneDrive-JohnsHopkins/Documents/Meetings/ArcticOceanDynamics_Nov25/ArcticBoundaryCurrentModel/nuPGCM/matrices/evolution_bowl3D_8.000000e-02_5.000000e-01.jld2' (0.013 GB)
└ @ nuPGCM /Users/twnh/Library/CloudStorage/OneDrive-JohnsHopkins/Documents/Meetings/ArcticOceanDynamics_Nov25/ArcticBoundaryCurrentModel/nuPGCM/src/matrices.jl:195


#### Re-order dofs

In [14]:
A_adv  =  A_adv[fe_data.dofs.p_b, fe_data.dofs.p_b]
A_diff = A_diff[fe_data.dofs.p_b, fe_data.dofs.p_b]
B_diff = B_diff[fe_data.dofs.p_b, :]
b_diff = b_diff[fe_data.dofs.p_b] ;

#### Preconditioners `P_diff` and `P_adv`

In [15]:
if typeof(arch) == CPU 
    P_diff = lu(A_diff)
    P_adv  = lu(A_adv)
else
    P_diff = Diagonal(on_architecture(arch, Vector(1 ./ diag(A_diff))))
    P_adv  = Diagonal(on_architecture(arch, Vector(1 ./ diag(A_adv))))
end
nothing

#### Move to architecture

In [16]:
A_adv  = on_architecture(arch, A_adv)
A_diff = on_architecture(arch, A_diff)
B_diff = on_architecture(arch, B_diff)
b_diff = on_architecture(arch, b_diff) ;

#### Setup evolution toolkit `evolution_toolkit`

In [17]:
evolution_toolkit = EvolutionToolkit(A_adv, P_adv, A_diff, P_diff, B_diff, b_diff) ;

#### Put it all together in the `model` struct

In [18]:
model = rest_state_model(arch, params, fe_data, inversion_toolkit, evolution_toolkit) ;

#### Set initial buoyancy

In [19]:
set_b!(model, x->b₀(x))
invert!(model) # sync flow with initial condition
save_vtk(model, ofile=@sprintf("%s/data/state_%016d.vtu", out_dir, 0))

┌ Info: VTK state saved to '/Users/twnh/Library/CloudStorage/OneDrive-JohnsHopkins/Documents/Meetings/ArcticOceanDynamics_Nov25/ArcticBoundaryCurrentModel/nuPGCM//data/state_0000000000000000.vtu'
└ @ nuPGCM /Users/twnh/Library/CloudStorage/OneDrive-JohnsHopkins/Documents/Meetings/ArcticOceanDynamics_Nov25/ArcticBoundaryCurrentModel/nuPGCM/src/IO.jl:32


### Solve

In [20]:
n_steps = Int(round(Tf / Δt))
n_save = n_steps ÷ 100
@time run!(model; n_steps, n_save)
println("Done.")

┌ Info: Beginning integration with
│   n_steps = 625
│   i_step = 1
│   n_save = 6
│   n_plot = Inf
│   n_info = 6
└ @ nuPGCM /Users/twnh/Library/CloudStorage/OneDrive-JohnsHopkins/Documents/Meetings/ArcticOceanDynamics_Nov25/ArcticBoundaryCurrentModel/nuPGCM/src/model.jl:78
┌ Info: t = 0.150000 (i = 6/625, Δt = 0.025000)
│ time elapsed: 00:00:03
│ estimated time remaining: 00:05:30
│ |u|ₘₐₓ = 1.0e-03, -3.0e-02 ≤ b ≤ 3.3e-02
└ @ nuPGCM /Users/twnh/Library/CloudStorage/OneDrive-JohnsHopkins/Documents/Meetings/ArcticOceanDynamics_Nov25/ArcticBoundaryCurrentModel/nuPGCM/src/model.jl:100
┌ Info: Model state saved to '/Users/twnh/Library/CloudStorage/OneDrive-JohnsHopkins/Documents/Meetings/ArcticOceanDynamics_Nov25/ArcticBoundaryCurrentModel/nuPGCM//data/state_0000000000000006.jld2'
└ @ nuPGCM /Users/twnh/Library/CloudStorage/OneDrive-JohnsHopkins/Documents/Meetings/ArcticOceanDynamics_Nov25/ArcticBoundaryCurrentModel/nuPGCM/src/IO.jl:5
┌ Info: VTK state saved to '/Users/twnh/Library/Cloud

 80.450319 seconds (280.16 M allocations: 29.282 GiB, 1.59% gc time, 4.83% compilation time: <1% of which was recompilation)
Done.

┌ Info: Model state saved to '/Users/twnh/Library/CloudStorage/OneDrive-JohnsHopkins/Documents/Meetings/ArcticOceanDynamics_Nov25/ArcticBoundaryCurrentModel/nuPGCM//data/state_0000000000000624.jld2'
└ @ nuPGCM /Users/twnh/Library/CloudStorage/OneDrive-JohnsHopkins/Documents/Meetings/ArcticOceanDynamics_Nov25/ArcticBoundaryCurrentModel/nuPGCM/src/IO.jl:5
┌ Info: VTK state saved to '/Users/twnh/Library/CloudStorage/OneDrive-JohnsHopkins/Documents/Meetings/ArcticOceanDynamics_Nov25/ArcticBoundaryCurrentModel/nuPGCM//data/state_0000000000000624.vtu'
└ @ nuPGCM /Users/twnh/Library/CloudStorage/OneDrive-JohnsHopkins/Documents/Meetings/ArcticOceanDynamics_Nov25/ArcticBoundaryCurrentModel/nuPGCM/src/IO.jl:32
